# Recursividad en Python 

Objetivos:
- entender el patrón de la recursividad,
- ver ejemplos clásicos,
- conectar con ejemplos útiles para física,
- comparar un método recursivo costoso con un método numérico más eficiente.

## 1. Importaciones mínimas

Usaremos únicamente módulos de la biblioteca estándar:
- `functools.cache` para optimizar el código usando decoradores de cache,
- `time` para medir tiempo,
- `tracemalloc` para una medida simple de memoria Python,
- `random` para generar matrices de prueba,
- `collections.Counter` para contar trabajo interno.

In [22]:
from functools import cache
from collections import Counter
import time
import tracemalloc
import random
import math

## 2. Una función simple para medir rendimiento

No buscaremos una evaluación “de laboratorio”, sino una comparación clara para estudiantes.
Mediremos:
- tiempo en segundos,
- memoria pico aproximada en KiB,
- resultado de la función.

In [23]:
def medir(func, *args, repetir=3, setup=None, **kwargs):
    tiempos = []
    memorias = []
    resultado = None

    for _ in range(repetir):
        if setup is not None:
            setup()

        tracemalloc.start()
        t0 = time.perf_counter()
        resultado = func(*args, **kwargs)
        t1 = time.perf_counter()
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        tiempos.append(t1 - t0)
        memorias.append(pico / 1024)

    return {
        'resultado': resultado,
        'tiempo_min_s': min(tiempos),
        'tiempo_prom_s': sum(tiempos) / len(tiempos),
        'memoria_pico_kib': max(memorias),
    }


def imprimir_fila(*cols, widths=None):
    if widths is None:
        widths = [12] * len(cols)
    texto = ' | '.join(str(c).ljust(w) for c, w in zip(cols, widths))
    print(texto)

## 3. Calentamiento: pensar recursivamente

Toda función recursiva necesita:
1. un **caso base**,
2. una llamada hacia un problema más pequeño.

In [26]:
def factorial(n):
    if n < 0:
        raise ValueError('n debe ser no negativo')
    if n in (0, 1):
        return 1
    return n * factorial(n - 1)


def suma_anidada(obj):
    if isinstance(obj, (int, float)):
        return obj
    total = 0
    for x in obj:
        total += suma_anidada(x)
    return total

print('factorial(6) =', factorial(6))
print('suma_anidada=', suma_anidada([[3], [[2], 3], (4, [5, 6])]))

factorial(6) = 720
suma_anidada= 23


## 4. Torres de Hanoi

Es un ejemplo clásico. Aquí usamos un **generador** para producir movimientos uno a uno.

In [19]:
def hanoi_lista(n, origen='A', auxiliar='B', destino='C'):
    if n <= 0:
        return []
    if n == 1:
        return [(origen, destino)]
    
    movimientos_1 = hanoi_lista(n - 1, origen, destino, auxiliar)
    movimiento_central = [(origen, destino)]
    movimientos_2 = hanoi_lista(n - 1, auxiliar, origen, destino)
    
    return movimientos_1 + movimiento_central + movimientos_2


movimientos = hanoi_lista(3)

print('Movimientos para 3 discos:')
for i, mov in enumerate(movimientos, start=1):
    print(i, mov)
print('Total =', len(movimientos))

Movimientos para 3 discos:
1 ('A', 'C')
2 ('A', 'B')
3 ('C', 'B')
4 ('A', 'C')
5 ('B', 'A')
6 ('B', 'C')
7 ('A', 'C')
Total = 7


In [20]:
def hanoi_acum(n, origen='A', auxiliar='B', destino='C', movimientos=None):
    if movimientos is None:
        movimientos = []
    
    if n <= 0:
        return movimientos
    
    if n == 1:
        movimientos.append((origen, destino))
        return movimientos
    
    hanoi_acum(n - 1, origen, destino, auxiliar, movimientos)
    movimientos.append((origen, destino))
    hanoi_acum(n - 1, auxiliar, origen, destino, movimientos)
    
    return movimientos


movimientos = hanoi_acum(3)

print('Movimientos para 3 discos:')
for i, mov in enumerate(movimientos, start=1):
    print(i, mov)
print('Total =', len(movimientos))

Movimientos para 3 discos:
1 ('A', 'C')
2 ('A', 'B')
3 ('C', 'B')
4 ('A', 'C')
5 ('B', 'A')
6 ('B', 'C')
7 ('A', 'C')
Total = 7


In [21]:
def hanoi_gen(n, origen='A', auxiliar='B', destino='C'):
    if n <= 0:
        return
    if n == 1:
        yield (origen, destino)
        return
    
    yield from hanoi_gen(n - 1, origen, destino, auxiliar)
    yield (origen, destino)
    yield from hanoi_gen(n - 1, auxiliar, origen, destino)


movimientos = list(hanoi_gen(3))

print('Movimientos para 3 discos:')
for i, mov in enumerate(movimientos, start=1):
    print(i, mov)
print('Total =', len(movimientos))

Movimientos para 3 discos:
1 ('A', 'C')
2 ('A', 'B')
3 ('C', 'B')
4 ('A', 'C')
5 ('B', 'A')
6 ('B', 'C')
7 ('A', 'C')
Total = 7


## 5. Fibonacci: ingenuo, con caché y como generador

Este ejemplo es perfecto para mostrar cuándo una recursión se vuelve costosa.

In [10]:
def fib_rec(n, metrics=None):
    if metrics is not None:
        metrics['llamadas'] += 1
    if n < 2:
        return n
    return fib_rec(n - 1, metrics) + fib_rec(n - 2, metrics)


@cache
def fib_cache(n):
    if n < 2:
        return n
    return fib_cache(n - 1) + fib_cache(n - 2)


def limpiar_fib_cache():
    fib_cache.cache_clear()


def fib_gen(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

metricas_fib = Counter()
print('fib_rec(10) =', fib_rec(10, metricas_fib), '| llamadas =', metricas_fib['llamadas'])
limpiar_fib_cache()
print('fib_cache(10) =', fib_cache(10), '| cache_info =', fib_cache.cache_info())
print('Primeros 10 Fibonacci con generador =', list(fib_gen(10)))

fib_rec(10) = 55 | llamadas = 177
fib_cache(10) = 55 | cache_info = CacheInfo(hits=8, misses=11, maxsize=None, currsize=11)
Primeros 10 Fibonacci con generador = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## 6. Ejemplo atractivo para físicos: bisección recursiva

La bisección no suele enseñarse primero como recursiva, pero hacerlo así ayuda a ver cómo un intervalo grande se reduce paso a paso.

Buscaremos una raíz de:
\[
\cos(x)-x = 0.
\]

In [11]:
def biseccion_rec(f, a, b, tol=1e-10, max_iter=100, depth=0):
    fa = f(a)
    fb = f(b)
    if fa * fb > 0:
        raise ValueError('El intervalo no encierra una raíz')

    c = (a + b) / 2
    fc = f(c)

    if abs(fc) < tol or (b - a) / 2 < tol or depth >= max_iter:
        return c

    if fa * fc < 0:
        return biseccion_rec(f, a, c, tol, max_iter, depth + 1)
    return biseccion_rec(f, c, b, tol, max_iter, depth + 1)

raiz = biseccion_rec(lambda x: math.cos(x) - x, 0, 1)
print('Raíz aproximada de cos(x) - x = 0:', raiz)

Raíz aproximada de cos(x) - x = 0: 0.7390851331874728


## 7. Otro ejemplo útil para física: polinomios de Legendre

Usamos la relación de recurrencia
\[
P_n(x)=
rac{(2n-1)xP_{n-1}(x)-(n-1)P_{n-2}(x)}{n}.
\]

Aquí la implementación es recursiva y usa caché para no recalcular.

In [12]:
@cache
def legendre(n, x):
    if n == 0:
        return 1.0
    if n == 1:
        return x
    return ((2*n - 1)*x*legendre(n - 1, x) - (n - 1)*legendre(n - 2, x)) / n

for n in range(6):
    print(f'P_{n}(0.3) = {legendre(n, 0.3)}')

P_0(0.3) = 1.0
P_1(0.3) = 0.3
P_2(0.3) = -0.365
P_3(0.3) = -0.3825
P_4(0.3) = 0.07293749999999999
P_5(0.3) = 0.34538625


## 8. Determinante por definición recursiva

Si una matriz está representada como una **lista de listas**, podemos calcular el determinante usando expansión por cofactores.

Esto es didácticamente muy valioso, aunque computacionalmente es caro.

In [13]:
def es_cuadrada(M):
    return len(M) > 0 and all(len(fila) == len(M) for fila in M)


def menor(M, fila, col):
    return [
        [M[i][j] for j in range(len(M)) if j != col]
        for i in range(len(M))
        if i != fila
    ]


def det_recursivo(M, metrics=None):
    if not es_cuadrada(M):
        raise ValueError('La matriz debe ser cuadrada y no vacía')

    n = len(M)

    if metrics is not None:
        metrics['llamadas'] += 1

    if n == 1:
        return M[0][0]
    if n == 2:
        if metrics is not None:
            metrics['casos_base_2x2'] += 1
        return M[0][0]*M[1][1] - M[0][1]*M[1][0]

    total = 0
    for j, a in enumerate(M[0]):
        if metrics is not None:
            metrics['cofactores'] += 1
            metrics['menores_construidos'] += 1
        total += ((-1) ** j) * a * det_recursivo(menor(M, 0, j), metrics)
    return total

## 9. Determinante recursivo con caché

Para usar `@cache`, la matriz debe convertirse a una estructura hashable, por ejemplo una tupla de tuplas.

In [14]:
def a_tuplas(M):
    return tuple(tuple(fila) for fila in M)


def limpiar_det_cache():
    det_cache.cache_clear()
    det_cache_metrics.clear()


det_cache_metrics = Counter()


@cache
def det_cache(Mt):
    n = len(Mt)
    det_cache_metrics['llamadas'] += 1

    if n == 1:
        return Mt[0][0]
    if n == 2:
        det_cache_metrics['casos_base_2x2'] += 1
        return Mt[0][0]*Mt[1][1] - Mt[0][1]*Mt[1][0]

    total = 0
    for j, a in enumerate(Mt[0]):
        det_cache_metrics['cofactores'] += 1
        sub = tuple(
            tuple(fila[k] for k in range(n) if k != j)
            for fila in Mt[1:]
        )
        total += ((-1) ** j) * a * det_cache(sub)
    return total

## 10. Determinante numérico por eliminación tipo LU

Aquí implementamos una versión sencilla con **pivoteo parcial** usando solo listas de Python.

Idea:
- transformar la matriz a triangular superior,
- corregir por intercambios de filas,
- multiplicar la diagonal.

Este método es muchísimo más eficiente que la expansión por cofactores.

In [15]:
def copiar_matriz(M):
    return [fila[:] for fila in M]


def det_lu_simple(M, metrics=None, tol=1e-12):
    if not es_cuadrada(M):
        raise ValueError('La matriz debe ser cuadrada y no vacía')

    A = copiar_matriz(M)
    n = len(A)
    signo = 1

    for k in range(n):
        pivote = max(range(k, n), key=lambda i: abs(A[i][k]))

        if abs(A[pivote][k]) < tol:
            return 0.0

        if pivote != k:
            A[k], A[pivote] = A[pivote], A[k]
            signo *= -1
            if metrics is not None:
                metrics['swaps'] += 1

        for i in range(k + 1, n):
            factor = A[i][k] / A[k][k]
            if metrics is not None:
                metrics['divisiones'] += 1
            for j in range(k + 1, n):
                A[i][j] -= factor * A[k][j]
                if metrics is not None:
                    metrics['mult_sub'] += 1
            A[i][k] = 0.0

    det = signo
    for i in range(n):
        det *= A[i][i]
        if metrics is not None:
            metrics['mult_diag'] += 1
    return det

## 11. Verificación rápida en una matriz pequeña

In [16]:
A = [
    [2, 1, 3],
    [0, -1, 4],
    [5, 2, 0],
]

m_rec = Counter()
d1 = det_recursivo(A, m_rec)

limpiar_det_cache()
d2 = det_cache(a_tuplas(A))
info_cache = det_cache.cache_info()

m_lu = Counter()
d3 = det_lu_simple(A, m_lu)

print('Matriz A =')
for fila in A:
    print(fila)
print()
print('det_recursivo =', d1)
print('det_cache     =', d2)
print('det_lu_simple =', d3)
print()
print('Métricas recursivo:', dict(m_rec))
print('Métricas cache    :', dict(det_cache_metrics))
print('cache_info        :', info_cache)
print('Métricas LU       :', dict(m_lu))

Matriz A =
[2, 1, 3]
[0, -1, 4]
[5, 2, 0]

det_recursivo = 19
det_cache     = 19
det_lu_simple = 19.0

Métricas recursivo: {'llamadas': 4, 'cofactores': 3, 'menores_construidos': 3, 'casos_base_2x2': 3}
Métricas cache    : {'llamadas': 4, 'cofactores': 3, 'casos_base_2x2': 3}
cache_info        : CacheInfo(hits=0, misses=4, maxsize=None, currsize=4)
Métricas LU       : {'swaps': 1, 'divisiones': 3, 'mult_sub': 5, 'mult_diag': 3}


## 12. Función para generar matrices aleatorias pequeñas

Usaremos enteros pequeños para que los determinantes no crezcan demasiado y para hacer la comparación manejable en clase.

In [17]:
random.seed(2026)


def matriz_aleatoria(n, a=-4, b=4):
    M = []
    for _ in range(n):
        fila = [random.randint(a, b) for _ in range(n)]
        M.append(fila)
    return M

for fila in matriz_aleatoria(4):
    print(fila)

[-3, 1, 4, 4]
[-3, -1, 4, 2]
[4, 3, 3, -1]
[-4, -3, -3, 0]


## 13. Comparación simple de rendimiento

La idea aquí no es hacer un benchmark profesional sino mostrar una tendencia clara.

Compararemos:
- determinante recursivo por definición,
- determinante recursivo con caché,
- determinante por eliminación tipo LU.

Para no demorar demasiado la clase, usamos tamaños pequeños.

In [18]:
def comparar_determinantes(n_min=2, n_max=7):
    encabezados = ['n', 'metodo', 'det', 'tiempo_min (s)', 'memoria (KiB)', 'trabajo interno']
    widths = [4, 18, 12, 16, 14, 24]
    imprimir_fila(*encabezados, widths=widths)
    print('-' * 96)

    for n in range(n_min, n_max + 1):
        M = matriz_aleatoria(n)

        # recursivo ingenuo
        met_rec = Counter()
        b_rec = medir(det_recursivo, M, repetir=1, metrics=met_rec)
        trabajo_rec = f"llamadas={met_rec['llamadas']}"
        imprimir_fila(n, 'recursivo', round(b_rec['resultado'], 6),
                      f"{b_rec['tiempo_min_s']:.6f}",
                      f"{b_rec['memoria_pico_kib']:.2f}",
                      trabajo_rec,
                      widths=widths)

        # recursivo con cache
        limpiar_det_cache()
        b_cache = medir(det_cache, a_tuplas(M), repetir=1, setup=limpiar_det_cache)
        info = det_cache.cache_info()
        trabajo_cache = f"misses={info.misses}, hits={info.hits}"
        imprimir_fila(n, 'rec + cache', round(b_cache['resultado'], 6),
                      f"{b_cache['tiempo_min_s']:.6f}",
                      f"{b_cache['memoria_pico_kib']:.2f}",
                      trabajo_cache,
                      widths=widths)

        # LU simple
        met_lu = Counter()
        b_lu = medir(det_lu_simple, M, repetir=1, metrics=met_lu)
        trabajo_lu = f"ops~{met_lu['divisiones'] + met_lu['mult_sub'] + met_lu['mult_diag']}"
        imprimir_fila(n, 'LU simple', round(b_lu['resultado'], 6),
                      f"{b_lu['tiempo_min_s']:.6f}",
                      f"{b_lu['memoria_pico_kib']:.2f}",
                      trabajo_lu,
                      widths=widths)
        print('-' * 96)

comparar_determinantes(2, 7)

n    | metodo             | det          | tiempo_min (s)   | memoria (KiB)  | trabajo interno         
------------------------------------------------------------------------------------------------
2    | recursivo          | 3            | 0.000063         | 0.52           | llamadas=1              
2    | rec + cache        | 3            | 0.000009         | 0.14           | misses=1, hits=0        
2    | LU simple          | 3.0          | 0.000527         | 0.61           | ops~4                   
------------------------------------------------------------------------------------------------
3    | recursivo          | -13          | 0.000313         | 1.44           | llamadas=4              
3    | rec + cache        | -13          | 0.000064         | 1.22           | misses=4, hits=0        
3    | LU simple          | -13.0        | 0.000055         | 0.65           | ops~11                  
------------------------------------------------------------------------------